# Simulation checkpointing command

Example of an in-headset command to save the current state of the simulation so that it can be revisited later:

<video src="../figures/checkpoint-example.webm" controls>

## Simulation and server setup

We start with the usual server setup, using a pre-bundled OpenMM simulation of a methane + nanotube system:

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

nanotube_simulation = OpenMMSimulation.from_xml_path("../systems/openmm/nanotube.openmm.zip")

imd_runner = OmniRunner.with_basic_server(nanotube_simulation, name='simulation checkpoints example')
imd_runner.load(0)
imd_runner.print_basic_info()

Serving "simulation checkpoints example" (ws://localhost:38801), discoverable on all interfaces on port 54545
Available simulations:
[0]: "nanotube.openmm"
Switched to [0]: "nanotube.openmm"


In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

## Defining a checkpointing command

We'll define a command that, when triggered, saves the current state of the current simulation and adds a user command to jump back to it:

In [3]:
CHECKPOINTS = []

def make_checkpoint():
    # checkpoint is openmm state
    state = nanotube_simulation.simulation.context.getState(
        getPositions=True,
        getVelocities=True,
        getParameters=True,
        getIntegratorParameters=True,
    )

    # add to last five checkpoints
    CHECKPOINTS.append(state)
    CHECKPOINTS[:] = CHECKPOINTS[-5:]

    # update the commands to jump between checkpoints
    for i, state in enumerate(CHECKPOINTS):
        def load_checkpoint(state=state):
            nanotube_simulation.simulation.context.setState(state)

        utilities.define_command(f"user/checkpoint/{i}", label=f"load checkpoint", icon=f"{i}", handler=load_checkpoint)


# define user command for making a checkpoint
utilities.define_command("user/checkpoint", label="make checkpoint", icon="🚩", handler=make_checkpoint)